# VoxForge Free Backend — TripoSR on Colab GPU (GRATIS)

1. Runtime → Change runtime type → pilih **T4 GPU** → Save
2. Klik tombol ▶️ di kiri cell (atau Runtime → Run all)
3. Tunggu sampai muncul `Running on public URL: https://xxxx.gradio.live`
4. Copy URL itu ke VoxForge (kotak 🆓 Mode Gratis — Colab URL)

Catatan: Colab tidur ~90 menit tidak aktif. Kalau mati, Run cell lagi.

In [ ]:
# ===== PIN NUMPY DULU (hindari downgrade oleh TripoSR) =====
!pip install -q "numpy==2.1.3" "numba==0.61.0"
!pip install -q "gradio==4.44.1" "torch==2.5.1" "transformers==4.45.2" "timm==1.0.11" "imageio" "imageio-ffmpeg" "trimesh"
!git clone https://github.com/VAST-AI-Research/TripoSR.git 2>/dev/null || echo 'already cloned'
%cd /content/TripoSR
# install TripoSR tapi JANGAN biarkan downgrade numpy
!pip install -q --no-deps -r requirements.txt

# ===== LOAD MODEL =====
import os, tempfile, trimesh
import gradio as gr
from PIL import Image
import torch
import numpy as np
print('numpy', np.__version__)
from tsr.system import TSR

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', device)
assert torch.cuda.is_available(), 'Pilih GPU di Runtime → Change runtime type!'

model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.renderer.set_chunk_size(8192)
model.to(device)
print('MODEL LOADED')

# ===== LAUNCH GRADIO =====
def image_to_glb(image):
    if image is None:
        return None
    pil = Image.fromarray(image).convert('RGB')
    with torch.no_grad():
        scene_codes = model([pil], device=device)
        meshes = model.extract_mesh(scene_codes, resolution=256, threshold=0.5)
        mesh = meshes[0]
    out = tempfile.NamedTemporaryFile(suffix='.glb', delete=False, dir='/tmp')
    mesh.export(out.name)
    return out.name

demo = gr.Interface(
    fn=image_to_glb,
    inputs=gr.Image(label='Upload gambar objek', type='numpy'),
    outputs=gr.Model3D(label='3D Model (GLB)'),
    title='VoxForge Free — TripoSR',
    description='Gratis via Colab GPU. Image-to-3D saja.'
)
demo.launch(share=True, server_name='0.0.0.0', server_port=7860, debug=True)